In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# LiNiO₂ on ECHIDNA

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the layered oxide LiNiO₂ to constant-wavelength neutron powder data from ECHIDNA at ANSTO.
Li and Ni mix between two sites, held to full occupancy by constraints, and the profile has
Finger-Cox-Jephcoat asymmetry.

It builds the catalogue case `pd-neut-cwl_lmo-echidna` step by step: the same project the command
line fits and the app's Examples open. Li and Ni share their sites through five relations
(displacements and occupancies).

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='pd-neut-cwl_lmo-echidna')

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('pd-neut-cwl_lmo-echidna-lmo', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['lmo']

Data 'pd-neut-cwl_lmo-echidna-lmo' is at data/pd-neut-cwl_lmo-echidna-lmo.edi


In [5]:
project.display.structure('lmo')

View(html=<4,406,565 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('pd-neut-cwl_lmo-echidna-echidna-data', destination='data')
project.experiments.add_from_data_path(
    name='echidna',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['echidna']
experiment.neutron_scattering_length = 'sears1992'
experiment.peak.type = 'cwl-tch-pseudo-voigt-fcj'
experiment.absorption.type = 'cylinder-hewat'
experiment.instrument.name = 'ECHIDNA'
experiment.instrument.facility = 'ANSTO'
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.3
experiment.peak.broad_gauss_w = 0.4
experiment.peak.broad_lorentz_y = 0.1
experiment.peak.asym_fcj_1 = 0.08
experiment.peak.asym_fcj_2 = 0.08
experiment.peak.cutoff_fwhm = 12
experiment.instrument.setup_wavelength = 1.6215
experiment.instrument.calib_sample_displacement = 0
experiment.instrument.calib_sample_transparency = 0
experiment.absorption.mu_r = 0.3
experiment.excluded_regions.create(start=0, end=12)
experiment.excluded_regions.create(start=162, end=180)
experiment.linked_structures.create(structure_id='lmo', scale=10)
experiment.background.create(position=12, intensity=550.0)
experiment.background.create(position=30, intensity=550.0)
experiment.background.create(position=50, intensity=550.0)
experiment.background.create(position=70, intensity=550.0)
experiment.background.create(position=90, intensity=550.0)
experiment.background.create(position=110, intensity=550.0)
experiment.background.create(position=130, intensity=550.0)
experiment.background.create(position=150, intensity=550.0)
experiment.background.create(position=162, intensity=550.0)

Data 'pd-neut-cwl_lmo-echidna-echidna-data' is at data/pd-neut-cwl_lmo-echidna-echidna.xye


The constraints the fit keeps, each between two named parameters (aliases).

In [7]:
project.analysis.aliases.create(id='biso_Li1', param=structure.atom_sites['Li1'].adp_iso)
project.analysis.aliases.create(id='biso_Li2', param=structure.atom_sites['Li2'].adp_iso)
project.analysis.aliases.create(id='biso_Ni1', param=structure.atom_sites['Ni1'].adp_iso)
project.analysis.aliases.create(id='biso_Ni2', param=structure.atom_sites['Ni2'].adp_iso)
project.analysis.aliases.create(id='occ_Li1', param=structure.atom_sites['Li1'].occupancy)
project.analysis.aliases.create(id='occ_Li2', param=structure.atom_sites['Li2'].occupancy)
project.analysis.aliases.create(id='occ_Ni1', param=structure.atom_sites['Ni1'].occupancy)
project.analysis.aliases.create(id='occ_Ni2', param=structure.atom_sites['Ni2'].occupancy)
project.analysis.constraints.create(id='1', expression='biso_Ni1 = biso_Li1')
project.analysis.constraints.create(id='2', expression='biso_Li2 = biso_Ni2')
project.analysis.constraints.create(id='3', expression='occ_Ni1 = 1 - occ_Li1')
project.analysis.constraints.create(id='4', expression='occ_Li2 = 1 - occ_Li1')
project.analysis.constraints.create(id='5', expression='occ_Ni2 = occ_Li1')

In [8]:
project.display.constraints()

View(html=<82,683 characters>, height=460)

In [9]:
project.display.pattern('echidna')

View(html=<454,947 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, 2θ zero and cell

The scale, the zero shift and the two cell lengths put the peaks in place and at the right
height.

In [10]:
experiment.linked_structures['lmo'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_c.free = True

In [11]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,lmo,cell,,length_a,2.88,0,0,99,Å
2,lmo,cell,,length_c,14.18,0,0,99,Å
3,echidna,instrument,,calib_twotheta_offset,0,0,-inf,inf,deg
4,echidna,linked_structure,lmo,scale,10,0,-inf,inf,


In [12]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 4 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       29.38        81.99
      1      0.009      8.05         6.16   92.5% ↓
  done · converged · 1 iters · reduced χ² 6.16 · Rwp 8.05 % · 0.01 s


View(html=<88,171 characters>, height=376)

In [13]:
project.display.pattern('echidna')

View(html=<456,547 characters>, height=480)

### Sample displacement and transparency

The sample sits a little off the diffractometer axis and the beam goes into it; both shift the
peaks with angle, unlike the zero.

In [14]:
experiment.instrument.calib_sample_displacement.free = True
experiment.instrument.calib_sample_transparency.free = True

In [15]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 6 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.05         6.16
      1      0.010      8.04         6.14   0.3% ↓
  done · converged · 1 iters · reduced χ² 6.14 · Rwp 8.04 % · 0.01 s


View(html=<90,254 characters>, height=376)

In [16]:
project.display.pattern('echidna')

View(html=<456,677 characters>, height=480)

### Background

The nine background points follow the level between the peaks.

In [17]:
for point in experiment.background:
    point.intensity.free = True

In [18]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 15 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.04         6.16
      1      0.012      6.46         3.98   35.4% ↓
  done · converged · 1 iters · reduced χ² 3.98 · Rwp 6.46 % · 0.01 s


View(html=<102,732 characters>, height=376)

In [19]:
project.display.pattern('echidna')

View(html=<485,525 characters>, height=480)

### Peak widths

In [20]:
for parameter in (
    experiment.peak.broad_gauss_u,
    experiment.peak.broad_gauss_v,
    experiment.peak.broad_gauss_w,
    experiment.peak.broad_lorentz_y,
):
    parameter.free = True

In [21]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 19 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.46         3.99
      1      0.012      6.39         3.90   2.2% ↓
  done · converged · 1 iters · reduced χ² 3.90 · Rwp 6.39 % · 0.01 s


View(html=<108,513 characters>, height=376)

In [22]:
project.display.pattern('echidna')

View(html=<485,400 characters>, height=480)

### Oxygen position

O's height along c is the only coordinate the symmetry leaves free.

In [23]:
structure.atom_sites['O'].fract_z.free = True

In [24]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.39         3.90
      1      0.013      6.20         3.67   6.0% ↓
  done · converged · 1 iters · reduced χ² 3.67 · Rwp 6.20 % · 0.02 s


View(html=<109,886 characters>, height=376)

In [25]:
project.display.pattern('echidna')

View(html=<485,390 characters>, height=480)

### Displacements

Li1's, Ni2's and O's; Ni1 and Li2 follow by the relations.

In [26]:
structure.atom_sites['Li1'].adp_iso.free = True
structure.atom_sites['Ni2'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [27]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 23 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.20         3.67
      1      0.014      5.31         2.70   26.6% ↓
  done · converged · 1 iters · reduced χ² 2.70 · Rwp 5.31 % · 0.02 s


View(html=<113,946 characters>, height=376)

In [28]:
project.display.pattern('echidna')

View(html=<485,313 characters>, height=480)

### Li/Ni mixing

Last, Li1's occupancy, which sets how much Ni sits on the Li site; it moves with the
displacements, so it comes after them.

In [29]:
structure.atom_sites['Li1'].occupancy.free = True

In [30]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 24 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.31         2.70
      1      0.015      4.99         2.39   11.5% ↓
  done · converged · 1 iters · reduced χ² 2.39 · Rwp 4.99 % · 0.02 s


View(html=<115,009 characters>, height=376)

In [31]:
project.display.pattern('echidna')

View(html=<485,301 characters>, height=480)

In [32]:
project.display.fit.correlations()

,1,2,3,4
1 lmo.cell.length_a,,,,
2 echidna.instrument.calib_twotheta_offset,0.89,,,
3 echidna.instrument.calib_sample_displacement,-0.94,-0.90,,
4 echidna.instrument.calib_sample_transparency,-0.74,-0.95,0.75,


In [33]:
project.display.structure('lmo')

View(html=<4,419,663 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [34]:
project.report.save_html('pd-neut-cwl_lmo-echidna.html')

Saving report to 'pd-neut-cwl_lmo-echidna.html'


## Save the project

In [35]:
project.save_as('pd-neut-cwl_lmo-echidna')

Saving project to 'pd-neut-cwl_lmo-echidna'
